In [2]:
stations_in_basin = {
    "18040001": ["11273400"], # Middle San Joaquin-Lower Chowchilla
    "18040002": ["11274550"], # Lower San Joaquin River
    "18040003": ["11303500"], # San Joaquin Delta
    "18040006": [], # Upper San Joaquin
    "18040007": [], # Fresno River
    "18040008": ["11272500"],# Upper Merced
    "18040009": ["11290000"], # Upper Tuolumne
    "18040010": ["11303000"], # Upper Stanislaus
    "18040011": [], # Upper Calaveras
    "18040012": ["11325500"], # Upper Mokelumne
    "18040013": [], # Upper Cosumnes
    "18040014": ["11255575"] # Panoche-San Luis Reservoir
}

In [17]:
!pip install dataretrieval

%cd /content
!rm -rf rainfall_kf

!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd rainfall_kf

!pip install -e .

import pandas as pd
def find_longest_continuous_period(series: pd.Series):
    """
    Find the longest uninterrupted non-NaN daily period.

    Parameters
    ----------
    series : pd.Series
        Daily series with DatetimeIndex.

    Returns
    -------
    (start, end)
        pandas.Timestamp pair or (NaT, NaT)
    """

    valid = series.notna()

    if not valid.any():
        return pd.NaT, pd.NaT

    # contiguous blocks
    groups = (valid != valid.shift()).cumsum()

    best_len = 0
    best_start = pd.NaT
    best_end = pd.NaT

    for _, block in valid.groupby(groups):

        # skip NaN blocks
        if not block.iloc[0]:
            continue

        block_index = block.index

        length = len(block_index)

        if length > best_len:
            best_len = length
            best_start = block_index[0]
            best_end = block_index[-1]

    return best_start, best_end

/content
Cloning into 'rainfall_kf'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 318 (delta 99), reused 107 (delta 51), pack-reused 145 (from 1)
Receiving objects: 100% (318/318), 109.36 MiB | 55.85 MiB/s, done.
Resolving deltas: 100% (135/135), done.
/content/rainfall_kf
Obtaining file:///content/rainfall_kf
  Preparing metadata (setup.py) ... done
  Running setup.py develop for rainfall_kf


In [21]:
import numpy as np
import pandas as pd
import xarray as xr
import pyproj
from dataretrieval import nwis, waterdata

START_DATE = "2000-01-01"
END_DATE = "2020-12-31"
PARAM_CODE = "00060"  # discharge

full_time = pd.date_range(START_DATE,END_DATE,freq="D")

data_arrays = []
site_metadata = []

for l in stations_in_basin.values():
    for site_no in l:
        print(f"Downloading {site_no}")
        
        # Site metadata
        site_info_df, _ = nwis.get_info(sites=site_no)

        if len(site_info_df) == 0:
            print(f"  No site info for {site_no}")
            continue

        site_info = site_info_df.iloc[0]

        # Daily discharge
        df, _ = waterdata.get_daily(monitoring_location_id=f"USGS-{site_no}",parameter_code=PARAM_CODE,time=f"{START_DATE}/{END_DATE}")

        if len(df) == 0:
            print(f"  No discharge data")
            continue

        ts = (df[["time", "value"]].copy()) # Keep only time/value
        ts["time"] = pd.to_datetime(ts["time"])
        ts = ts.set_index("time")      
        ts["value"] = pd.to_numeric(ts["value"], errors="coerce") # convert to numeric

        series = (ts["value"].reindex(full_time).astype(np.float32)) # Reindex to full daily range

        if series.notna().sum() == 0: # Skip gauges with no valid observations
            print(f"  All NaN")
            continue

        # Continuous period
        continuous_start, continuous_end = (find_longest_continuous_period(series))

        # Create DataArray
        da = xr.DataArray(series.values,dims=["time"],coords={"time": full_time},name=site_no)

        data_arrays.append(da)

        # Metadata
        site_metadata.append({
            "site_no": site_no,
            "station_name": site_info.get("station_nm"),
            "basin_huc8": site_info.get("huc_cd"),
            "continuous_start": continuous_start,
            "continuous_end": continuous_end,
            "latitude": site_info.get("dec_lat_va"),
            "longitude": site_info.get("dec_long_va"),
            "altitude": site_info.get("alt_va"),
            "drainage_area": site_info.get("drain_area_va"),
            "coord_datum": site_info.get("dec_coord_datum_cd"),
            "altitude_datum": site_info.get("alt_datum_cd"),
            "map_nm": site_info.get("map_nm"),
            "map_scale_fc": site_info.get("map_scale_fc")
        })

    # Combine into Dataset
    streamflow = xr.concat(data_arrays, dim="site")
    streamflow = streamflow.assign_coords(site=[m["site_no"] for m in site_metadata])

    ds = xr.Dataset({"streamflow": streamflow})

    # Site coordinates
    ds = ds.assign_coords({
        "site_no": ("site", [m["site_no"] for m in site_metadata]),
        "station_name": ("site", [m["station_name"] for m in site_metadata]),
        "basin_huc8": ("site", [m["basin_huc8"] for m in site_metadata]),
        "continuous_start": ("site", [m["continuous_start"] for m in site_metadata]),
        "continuous_end": ("site", [m["continuous_end"] for m in site_metadata]),
        "latitude": ("site", [m["latitude"] for m in site_metadata]),
        "longitude": ("site", [m["longitude"] for m in site_metadata]),
        "altitude": ("site", [m["altitude"] for m in site_metadata]),
        "drainage_area": ("site", [m["drainage_area"] for m in site_metadata]),
        "coord_datum": ("site", [m["coord_datum"] for m in site_metadata]),
        "altitude_datum": ("site", [m["altitude_datum"] for m in site_metadata]),
        "map_nm": ("site", [m["map_nm"] for m in site_metadata]),
        "map_scale_fc": ("site", [m["map_scale_fc"] for m in site_metadata])
    })

    # Attributes
    ds["streamflow"].attrs = {
        "units": "ft^3/s",
        "parameter_code": PARAM_CODE,
        "description": "Daily mean stream discharge"
    }

    ds.attrs = {
        "source": "USGS NWIS",
        "time_period": f"{START_DATE} to {END_DATE}",
        "parameter_code": PARAM_CODE
    }

    ds["drainage_area"].attrs = {"units": "square miles"}

/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


/tmp/ipykernel_33118/1250997683.py:21: DeprecationWarning: `nwis.get_info` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_monitoring_locations()` instead.
  site_info_df, _ = nwis.get_info(sites=site_no)


In [19]:
ds

<xarray.Dataset> Size: 311kB
Dimensions:           (time: 7671, site: 8)
Coordinates: (12/16)
  * time              (time) datetime64[ns] 61kB 2000-01-01 ... 2020-12-31
  * site              (site) <U8 256B '11273400' '11274550' ... '11255575'
    site_no           (site) <U8 256B '11273400' '11274550' ... '11255575'
    station_name      (site) <U38 1kB 'SAN JOAQUIN R AB MERCED R NR NEWMAN CA...
    basin_huc8        (site) int64 64B 18040001 18040002 ... 18040012 18040014
    continuous_start  (site) datetime64[ns] 64B 2011-07-17 ... 2003-12-01
    ...                ...
    drainage_area     (site) int64 64B 7949 9694 13539 1273 1884 1075 661 305
    geometry          (site) object 64B POINT (-120.9761399 37.34718815) ... ...
    coord_datum       (site) <U5 160B 'NAD83' 'NAD83' ... 'NAD83' 'NAD83'
    altitude_datum    (site) <U6 192B 'NAVD88' 'NAVD88' ... 'NGVD29' 'NAVD88'
    map_nm            (site) <U32 1kB 'GUSTINE' ... 'CHOUNET RANCH, CA'
    map_scale_fc      (site) float64 64B 2.4e+04 2.4e+04 2.4e+04 ... nan 2.4e+04
Data variables:
    streamflow        (site, time) float32 245kB nan nan nan nan ... 0.0 0.0 0.0
Attributes:
    source:          USGS NWIS
    time_period:     2000-01-01 to 2020-12-31
    parameter_code:  00060

In [22]:
ds.to_netcdf("/content/rainfall_kf/data/streamflow.nc")

In [1]:
# unit conversion factor
cfs_to_m3s = 0.0283168 # cfs to m^3/s